# Border Artifact Classifier

Part of **Exposing Dataset Artifacts in Medical AI**. This stage builds an explicit classifier for dataset/acquisition cues before evaluating their relationship to DR prediction.

> Dataset files are not distributed. Update `BASE_PATH` for your environment.


In [ ]:
from google.colab import drive
import os, shutil, random
from glob import glob
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Fundus_Artifact_Project'
BINARY_BORDER_DIR = os.path.join(BASE_PATH, 'Binary_Border_Classifier')
RESULTS_DIR = os.path.join(BASE_PATH, 'Results', 'border_artifact_model')
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
# Build balanced source pools used for the artifact-cue experiment
APTOS_IMG_DIR = os.path.join(BASE_PATH, 'APTOS_2019', 'train_images')
MESSIDOR_IMG_DIR = os.path.join(BASE_PATH, 'Messidor_2', 'my_preprocessed')
HAS_BORDER_DIR = os.path.join(BINARY_BORDER_DIR, 'has_border')
NO_BORDER_DIR = os.path.join(BINARY_BORDER_DIR, 'no_border')
for folder in [HAS_BORDER_DIR, NO_BORDER_DIR]:
    if os.path.exists(folder): shutil.rmtree(folder)
    os.makedirs(folder)
random.seed(42)
aptos_imgs = glob(f'{APTOS_IMG_DIR}/*.png')
messidor_imgs = glob(f'{MESSIDOR_IMG_DIR}/*.png')
n = min(500, len(aptos_imgs), len(messidor_imgs))
for path in random.sample(aptos_imgs, n): shutil.copy(path, HAS_BORDER_DIR)
for path in random.sample(messidor_imgs, n): shutil.copy(path, NO_BORDER_DIR)
print(f'Prepared {n} images per class')


In [ ]:
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
IMG_SIZE, BATCH_SIZE = 224, 32
transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
full_dataset = datasets.ImageFolder(BINARY_BORDER_DIR, transform=transform)
generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(full_dataset)); val_size = len(full_dataset)-train_size
train_set, val_set = random_split(full_dataset,[train_size,val_size],generator=generator)
train_loader = DataLoader(train_set,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_set,batch_size=BATCH_SIZE,shuffle=False)
print(full_dataset.class_to_idx)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
EPOCHS=10
train_acc=[]; val_acc=[]
for epoch in range(EPOCHS):
    model.train(); correct=total=0
    for inputs,labels in train_loader:
        inputs,labels=inputs.to(device),labels.to(device)
        optimizer.zero_grad(); outputs=model(inputs); loss=criterion(outputs,labels); loss.backward(); optimizer.step()
        correct += (outputs.argmax(1)==labels).sum().item(); total += labels.size(0)
    train_acc.append(correct/total)
    model.eval(); correct=total=0
    with torch.no_grad():
        for inputs,labels in val_loader:
            inputs,labels=inputs.to(device),labels.to(device); outputs=model(inputs)
            correct += (outputs.argmax(1)==labels).sum().item(); total += labels.size(0)
    val_acc.append(correct/total)
    print(f'Epoch {epoch+1}/{EPOCHS} | train={train_acc[-1]:.4f} | val={val_acc[-1]:.4f}')


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5)); plt.plot(train_acc,label='Train'); plt.plot(val_acc,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Border-cue classifier'); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(RESULTS_DIR,'accuracy_plot.png'),dpi=300,bbox_inches='tight'); plt.show()


In [ ]:
# Save trained weights; interpretability can be run with TorchCAM/Grad-CAM in a separate analysis pass.
model_path=os.path.join(RESULTS_DIR,'border_classifier_resnet18.pt')
torch.save(model.state_dict(),model_path)
print(model_path)


## Interpretation caution

The binary classes here are constructed from different source datasets. High accuracy therefore demonstrates learnable **source/acquisition cues**, not proof that borders alone are the causal feature. Attribution maps and controlled preprocessing comparisons are needed to isolate the shortcut mechanism.
